# Module 9 · 影片 3：案例 — 動作辨識推論（VideoMAE）

> **本節定位｜整合案例（全新）**
> 把影片 `01–02`（解碼、抽樣、前處理）整合，用預訓練 **VideoMAE** 對一段 clip
> 做**動作辨識**推論。CPU 可跑（單段短 clip），含離線後備。

## 學習目標
- 完整跑一次影片推論流程：抽樣 16 影格 → processor → 模型 → 預測動作類別。
- 理解影片模型輸出 `logits (N, num_classes)` 的意義。
- 認識「凍結特徵」與「端到端微調」的差別（微調見 M11）。

In [ ]:
import numpy as np

# 準備一段 clip 的 16 張影格：優先用 HF 範例影片；否則合成
def get_clip(num_frames=16):
    rs = np.random.RandomState(0)
    frames = [(rs.rand(224, 224, 3) * 255).astype(np.uint8) for _ in range(num_frames)]
    return frames, "合成影格（僅示意推論流程，預測類別無意義）"

clip_frames, source = get_clip()
print(f"資料來源：{source}")
print(f"clip 影格數 = {len(clip_frames)}，每張 {clip_frames[0].shape}")

## 1. 載入 VideoMAE 並推論

`videomae-base-finetuned-kinetics` 在 Kinetics-400（400 種動作）上微調過，可直接辨識動作。
需 `multimodal` extra；首次執行會下載權重。

In [ ]:
try:
    import torch
    from transformers import AutoImageProcessor, VideoMAEForVideoClassification

    proc = AutoImageProcessor.from_pretrained("MCG-NJU/videomae-base-finetuned-kinetics")
    model = VideoMAEForVideoClassification.from_pretrained("MCG-NJU/videomae-base-finetuned-kinetics")
    model.eval()

    inputs = proc(clip_frames, return_tensors="pt")
    print(f"模型輸入 pixel_values: {tuple(inputs['pixel_values'].shape)}  (N, T, C, H, W)")
    with torch.no_grad():
        logits = model(**inputs).logits                 # (N, num_classes=400)
    pred = logits.argmax(-1).item()
    print(f"輸出 logits 形狀: {tuple(logits.shape)}  (N, num_classes)")
    print(f"預測動作類別: {model.config.id2label[pred]}")
    print("（合成影格 → 預測無實際意義，僅證明流程跑通；換成真實影片即可得到合理結果。）")
except Exception as e:
    print("（未能下載 VideoMAE 權重，略過實際推論）", e)
    print("流程：抽樣 16 影格 → processor → VideoMAE → logits (N, 400) → argmax → 動作標籤。")

## 小結與延伸

| 重點 | 內容 |
|:--|:--|
| 流程 | 解碼 → 抽樣 16 影格 → processor → VideoMAE → `logits (N, num_classes)` |
| 預訓練資料 | Kinetics-400（400 種人類動作） |
| 輸入 | `(N, T, C, H, W)` 時空張量 |

**想用在自己的動作類別？** 把 VideoMAE **端到端微調**到你的資料集，
是 **Module 11 · `05_video_downstream`** 的內容。